# Particle Drift Dynamics — NN Inference Simulation

This notebook models 2 particles evolving under a **kernel-weighted drift field**, imitating the inference procedure of a neural network. Each step, particles are attracted toward target positions $y_1, y_2$ and repelled from each other, with optional friction ($\gamma$), noise ($\varepsilon$), and temperature-like scale ($\tau$).

---

### Kernel
$$k(x, y) = e^{-|x-y|/\tau}$$

### Drift for particle $x$ (attraction + inter-particle repulsion)
$$v^+(x) = \frac{k(x,y_1)(y_1 - x) + k(x,y_2)(y_2 - x)}{k(x,y_1) + k(x,y_2)}$$
$$v_1(x_1, x_2) = v^+(x_1) - k(x_1, x_2)(x_2 - x_1)$$

### Update rule (with friction and noise)
$$x_i \leftarrow x_i + (1-\gamma)\,v_i + \varepsilon\, \eta, \quad \eta \sim \text{Uniform}(-1,1)$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from ipywidgets import interact, FloatSlider, IntSlider, widgets
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

## Core Physics Functions

In [2]:
def kernel(x, y, tau=1.0):
    """Exponential kernel: k(x,y) = exp(-|x-y| / tau)"""
    return np.exp(-np.abs(x - y) / tau)


def vplus(x, x_other, y1, y2, tau=1.0):
    """Weighted attraction toward targets y1, y2 (softmax-like)."""
    w1 = kernel(x, y1, tau)
    w2 = kernel(x, y2, tau)
    denom = max(w1 + w2, 1e-300)
    return (w1 * (y1 - x) + w2 * (y2 - x)) / denom


def v1(x1, x2, y1, y2, tau=1.0):
    """Drift of particle 1: attraction to targets minus repulsion from particle 2."""
    return vplus(x1, x2, y1, y2, tau) - kernel(x1, x2, tau) * (x2 - x1)


def v2(x1, x2, y1, y2, tau=1.0):
    """Drift of particle 2: attraction to targets minus repulsion from particle 1."""
    return vplus(x2, x1, y1, y2, tau) - kernel(x2, x1, tau) * (x1 - x2)


def step(state, y1, y2, gamma=0.0, epsilon=0.0, tau=1.0):
    """Single discrete step for both particles."""
    x1, x2 = state
    noise1 = epsilon * np.random.uniform(-1, 1)
    noise2 = epsilon * np.random.uniform(-1, 1)
    new_x1 = x1 + (1 - gamma) * v1(x1, x2, y1, y2, tau) + noise1
    new_x2 = x2 + (1 - gamma) * v2(x1, x2, y1, y2, tau) + noise2
    return (new_x1, new_x2)


def simulate(x1_init, x2_init, y1, y2, n_steps, gamma=0.0, epsilon=0.0, tau=1.0):
    """Run the simulation; returns arrays of shape (n_steps+1,) for each particle."""
    trajectory = [(x1_init, x2_init)]
    state = (x1_init, x2_init)
    for _ in range(n_steps):
        state = step(state, y1, y2, gamma, epsilon, tau)
        trajectory.append(state)
    traj = np.array(trajectory)  # shape (n+1, 2)
    return traj[:, 0], traj[:, 1]

---

## Section 1 — Basic 2-Particle Evolution

Simplest model: no friction, no noise, no τ tuning. Particles evolve toward fixed targets using the basic kernel repulsion/attraction.

Sliders: **x1(0)**, **x2(0)**, **steps**

In [3]:
def plot_basic(x1_init=0.55, x2_init=-0.5, n=10):
    traj1, traj2 = simulate(x1_init, x2_init, y1=0.0, y2=0.0,
                             n_steps=n, gamma=0.0, epsilon=0.0, tau=1.0)
    steps = np.arange(n + 1)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(steps[1:], traj1[1:], color='royalblue', s=40, zorder=3, label='x1')
    ax.scatter(steps[1:], traj2[1:], color='crimson', s=40, zorder=3, label='x2')
    ax.set_xlabel('step n')
    ax.set_ylabel('x')
    ax.set_title(f'x1(0) = {x1_init:.2f}   x2(0) = {x2_init:.2f}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_basic,
    x1_init=FloatSlider(value=0.55, min=0, max=2, step=0.01, description='x1(0)', style={'description_width': '70px'}),
    x2_init=FloatSlider(value=-0.5, min=-2, max=0, step=0.01, description='x2(0)', style={'description_width': '70px'}),
    n=IntSlider(value=10, min=10, max=100, step=1, description='steps', style={'description_width': '70px'})
);

interactive(children=(FloatSlider(value=0.55, description='x1(0)', max=2.0, step=0.01, style=SliderStyle(descr…

---

## Section 2 — Single Particle with Friction and τ

One particle drifting under the field $v(x) = 2x e^{-2x/\tau} - x e^{-x/\tau}$, with friction scaling and temperature $\tau$. The dashed red line marks the fixed point at $\ln 2$.

Sliders: **x(0)**, **steps**, **friction γ**, **τ**

In [4]:
def v_single(x, tau):
    """Single-particle drift field: v(x) = 2x*exp(-2x/τ) - x*exp(-x/τ)"""
    return 2 * x * np.exp(-2 * x / tau) - x * np.exp(-x / tau)

def plot_single_particle(x0_init=0.5, n=10, gamma=0.0, tau=1.0):
    pts = [x0_init]
    x = x0_init
    for _ in range(n):
        x = x + (1 - gamma) * v_single(x, tau)
        pts.append(x)
    pts = np.array(pts)
    steps = np.arange(n + 1)

    fixed_pt = tau*np.log(2)  # fixed point of the map

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(steps, pts, color='royalblue', s=40, zorder=3)
    ax.axhline(fixed_pt, color='crimson', linestyle='--', linewidth=1.5,
               label=f'fixed point ln(2) ≈ {fixed_pt:.3f}')
    ax.set_xlabel('step n')
    ax.set_ylabel('x')
    ax.set_title(f'x(0) = {x0_init:.2f}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_single_particle,
    x0_init=FloatSlider(value=0.5, min=0, max=1, step=0.01, description='x(0)', style={'description_width': '80px'}),
    n=IntSlider(value=10, min=5, max=100, step=1, description='steps', style={'description_width': '80px'}),
    gamma=FloatSlider(value=0.0, min=0, max=1, step=0.01, description='friction γ', style={'description_width': '80px'}),
    tau=FloatSlider(value=1.0, min=0.01, max=2, step=0.01, description='τ', style={'description_width': '80px'})
);

interactive(children=(FloatSlider(value=0.5, description='x(0)', max=1.0, step=0.01, style=SliderStyle(descrip…

---

## Section 3 — 2 Particles with Targets and Friction

Particles evolve toward user-defined target positions $y_1, y_2$ with softmax-weighted attraction and inter-particle repulsion. Friction $\gamma$ damps the drift.

Sliders: **x1(0)**, **x2(0)**, **target y1**, **target y2**, **steps**, **friction γ**

In [5]:
def plot_2p_friction(x1_init=0.5, x2_init=-0.5, y1=0.1, y2=-0.1, n=20, gamma=0.0):
    traj1, traj2 = simulate(x1_init, x2_init, y1, y2,
                             n_steps=n, gamma=gamma, epsilon=0.0, tau=1.0)
    steps = np.arange(n + 1)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.scatter(steps[1:], traj1[1:], color='royalblue', s=40, zorder=3)
    ax.scatter(steps[1:], traj2[1:], color='crimson', s=40, zorder=3)
    ax.axhline(y1, color='gray', linestyle='--', linewidth=1.2, alpha=0.8)
    ax.axhline(y2, color='gray', linestyle='--', linewidth=1.2, alpha=0.8)

    ylim = ax.get_ylim()
    ax.text(0.88, 0.85, 'x1', transform=ax.transAxes, color='royalblue', fontweight='bold', fontsize=12)
    ax.text(0.88, 0.75, 'x2', transform=ax.transAxes, color='crimson', fontweight='bold', fontsize=12)
    ax.text(0.02, 0.55, 'y1', transform=ax.transAxes, color='gray', fontweight='bold', fontsize=12)
    ax.text(0.02, 0.45, 'y2', transform=ax.transAxes, color='gray', fontweight='bold', fontsize=12)

    ax.set_xlabel('step n')
    ax.set_ylabel('x')
    ax.set_title(f'x1(0) = {x1_init:.2f}   x2(0) = {x2_init:.2f}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_2p_friction,
    x1_init=FloatSlider(value=0.5,  min=0,   max=2,  step=0.01, description='x1(0)',     style={'description_width': '90px'}),
    x2_init=FloatSlider(value=-0.5, min=-2,  max=0,  step=0.01, description='x2(0)',     style={'description_width': '90px'}),
    y1=     FloatSlider(value=0.1,  min=0,   max=2,  step=0.01, description='target y1', style={'description_width': '90px'}),
    y2=     FloatSlider(value=-0.1, min=-2,  max=0,  step=0.01, description='target y2', style={'description_width': '90px'}),
    n=      IntSlider  (value=20,   min=5,   max=100,step=1,    description='steps',      style={'description_width': '90px'}),
    gamma=  FloatSlider(value=0.0,  min=0,   max=1,  step=0.01, description='friction γ', style={'description_width': '90px'})
);

interactive(children=(FloatSlider(value=0.5, description='x1(0)', max=2.0, step=0.01, style=SliderStyle(descri…

---

## Section 4 — 2 Particles with Friction + Noise

Adds stochastic noise $\varepsilon \cdot \text{Uniform}(-1,1)$ to each particle's update. Run multiple times to see different noise realizations.

Sliders: **x1(0)**, **x2(0)**, **target y1**, **target y2**, **steps**, **friction γ**, **noise ε**

In [6]:
def plot_2p_noise(x1_init=0.5, x2_init=-0.5, y1=0.1, y2=-0.1, n=20, gamma=0.0, epsilon=0.0):
    traj1, traj2 = simulate(x1_init, x2_init, y1, y2,
                             n_steps=n, gamma=gamma, epsilon=epsilon, tau=1.0)
    steps = np.arange(n + 1)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.scatter(steps[1:], traj1[1:], color='royalblue', s=40, zorder=3)
    ax.scatter(steps[1:], traj2[1:], color='crimson', s=40, zorder=3)
    ax.axhline(y1, color='gray', linestyle='--', linewidth=1.2, alpha=0.8)
    ax.axhline(y2, color='gray', linestyle='--', linewidth=1.2, alpha=0.8)

    ax.text(0.88, 0.85, 'x1', transform=ax.transAxes, color='royalblue', fontweight='bold', fontsize=12)
    ax.text(0.88, 0.75, 'x2', transform=ax.transAxes, color='crimson', fontweight='bold', fontsize=12)
    ax.text(0.02, 0.55, 'y1', transform=ax.transAxes, color='gray', fontweight='bold', fontsize=12)
    ax.text(0.02, 0.45, 'y2', transform=ax.transAxes, color='gray', fontweight='bold', fontsize=12)

    noise_str = f'ε = {epsilon:.2f}' if epsilon > 0 else 'no noise'
    ax.set_xlabel('step n')
    ax.set_ylabel('x')
    ax.set_title(f'x1(0) = {x1_init:.2f}   x2(0) = {x2_init:.2f}   [{noise_str}]')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_2p_noise,
    x1_init=FloatSlider(value=0.5,  min=0,   max=2,  step=0.01, description='x1(0)',     style={'description_width': '90px'}),
    x2_init=FloatSlider(value=-0.5, min=-2,  max=0,  step=0.01, description='x2(0)',     style={'description_width': '90px'}),
    y1=     FloatSlider(value=0.1,  min=0,   max=2,  step=0.01, description='target y1', style={'description_width': '90px'}),
    y2=     FloatSlider(value=-0.1, min=-2,  max=0,  step=0.01, description='target y2', style={'description_width': '90px'}),
    n=      IntSlider  (value=20,   min=5,   max=100,step=1,    description='steps',      style={'description_width': '90px'}),
    gamma=  FloatSlider(value=0.0,  min=0,   max=1,  step=0.01, description='friction γ', style={'description_width': '90px'}),
    epsilon=FloatSlider(value=0.0,  min=0,   max=1,  step=0.01, description='noise ε',    style={'description_width': '90px'})
);

interactive(children=(FloatSlider(value=0.5, description='x1(0)', max=2.0, step=0.01, style=SliderStyle(descri…

---

## Section 5 — Full Model with τ (no target range restrictions)

Same as above but with $\tau$ controlling the kernel width, and the full $[-2, 2]$ range for all positions. This is the most expressive variant before the final combined demo.

Sliders: **x1(0)**, **x2(0)**, **target y1**, **target y2**, **steps**, **friction γ**, **noise ε**, **τ**

In [ ]:
def plot_full_tau(x1_init=0.5, x2_init=-0.5, y1=0.1, y2=-0.1,
                  n=20, gamma=0.0, epsilon=0.0, tau=1.0):
    traj1, traj2 = simulate(x1_init, x2_init, y1, y2,
                             n_steps=n, gamma=gamma, epsilon=epsilon, tau=tau)
    steps = np.arange(n + 1)

    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    ax.scatter(steps[1:], traj1[1:], color='royalblue', s=45, zorder=3)
    ax.scatter(steps[1:], traj2[1:], color='crimson', s=45, zorder=3)
    ax.axhline(y1, color='gray', linestyle='--', linewidth=1.2, alpha=0.8)
    ax.axhline(y2, color='gray', linestyle='--', linewidth=1.2, alpha=0.8)

    ax.text(0.88, 0.85, 'x1', transform=ax.transAxes, color='royalblue', fontweight='bold', fontsize=12)
    ax.text(0.88, 0.75, 'x2', transform=ax.transAxes, color='crimson', fontweight='bold', fontsize=12)
    ax.text(0.02, 0.55, 'y1', transform=ax.transAxes, color='gray', fontweight='bold', fontsize=12)
    ax.text(0.02, 0.45, 'y2', transform=ax.transAxes, color='gray', fontweight='bold', fontsize=12)

    ax.set_xlabel('step n')
    ax.set_ylabel('x')
    ax.set_title(f'x1(0)={x1_init:.2f}  x2(0)={x2_init:.2f}  τ={tau:.2f}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_full_tau,
    x1_init=FloatSlider(value=0.5,  min=-2,  max=2,  step=0.01, description='x1(0)',     style={'description_width': '90px'}),
    x2_init=FloatSlider(value=-0.5, min=-2,  max=2,  step=0.01, description='x2(0)',     style={'description_width': '90px'}),
    y1=     FloatSlider(value=0.1,  min=-2,  max=2,  step=0.01, description='target y1', style={'description_width': '90px'}),
    y2=     FloatSlider(value=-0.1, min=-2,  max=2,  step=0.01, description='target y2', style={'description_width': '90px'}),
    n=      IntSlider  (value=20,   min=5,   max=100,step=1,    description='steps',      style={'description_width': '90px'}),
    gamma=  FloatSlider(value=0.0,  min=0,   max=1,  step=0.01, description='friction γ', style={'description_width': '90px'}),
    epsilon=FloatSlider(value=0.0,  min=0,   max=1,  step=0.01, description='noise ε',    style={'description_width': '90px'}),
    tau=    FloatSlider(value=1.0,  min=0.01,max=2,  step=0.01, description='τ',          style={'description_width': '90px'})
);

---

## Section 6 — Full Interactive Demo (Final Model)

Complete model matching the final Mathematica `Manipulate` cell: full $[-2,2]$ ranges, friction, noise, τ. This is the primary exploration tool.

Sliders: **x1(0)**, **x2(0)**, **target y1**, **target y2**, **steps**, **friction γ**, **noise ε**, **τ**

In [ ]:
def plot_combined(x1_init=0.5, x2_init=-0.5, y1=0.1, y2=-0.1,
                   n=20, gamma=0.0, epsilon=0.0, tau=1.0):
    traj1, traj2 = simulate(x1_init, x2_init, y1, y2,
                             n_steps=n, gamma=gamma, epsilon=epsilon, tau=tau)
    steps = np.arange(n + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # --- Left: trajectory plot ---
    ax = axes[0]
    ax.scatter(steps, traj1, color='royalblue', s=45, zorder=3)
    ax.scatter(steps, traj2, color='crimson', s=45, zorder=3)
    ax.axhline(y1, color='gray', linestyle='--', linewidth=1.4, alpha=0.9)
    ax.axhline(y2, color='gray', linestyle='--', linewidth=1.4, alpha=0.9)

    ax.text(0.88, 0.88, 'x1', transform=ax.transAxes, color='royalblue', fontweight='bold', fontsize=13)
    ax.text(0.88, 0.78, 'x2', transform=ax.transAxes, color='crimson',   fontweight='bold', fontsize=13)
    ax.text(0.02, 0.58, 'y1', transform=ax.transAxes, color='gray',       fontweight='bold', fontsize=13)
    ax.text(0.02, 0.46, 'y2', transform=ax.transAxes, color='gray',       fontweight='bold', fontsize=13)

    ax.set_xlabel('step n', fontsize=12)
    ax.set_ylabel('x', fontsize=12)
    ax.set_title(f'x1(0)={x1_init:.2f}  x2(0)={x2_init:.2f}', fontsize=12)
    ax.grid(True, alpha=0.3)

    # --- Right: phase portrait (x1 vs x2) ---
    ax2 = axes[1]
    ax2.scatter(traj1, traj2, c=steps, cmap='viridis', s=45, zorder=3)
    ax2.plot(traj1, traj2, color='gray', alpha=0.4, linewidth=1)
    ax2.scatter([x1_init], [x2_init], color='black', s=120, zorder=5, marker='o', label='start')
    ax2.scatter([traj1[-1]], [traj2[-1]], color='orange', s=120, zorder=5, marker='*', label='end')
    ax2.axvline(y1, color='royalblue', linestyle=':', alpha=0.6, linewidth=1.2, label=f'y1={y1:.2f}')
    ax2.axhline(y2, color='crimson',   linestyle=':', alpha=0.6, linewidth=1.2, label=f'y2={y2:.2f}')
    ax2.set_xlabel('x1', fontsize=12)
    ax2.set_ylabel('x2', fontsize=12)
    ax2.set_title('Phase portrait (x1 vs x2)', fontsize=12)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    fig.suptitle(f'γ={gamma:.2f}  ε={epsilon:.2f}  τ={tau:.2f}  steps={n}', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

interact(plot_combined,
    x1_init=FloatSlider(value=0.5,  min=-2,  max=2,  step=0.01, description='x1(0)',     style={'description_width': '95px'}),
    x2_init=FloatSlider(value=-0.5, min=-2,  max=2,  step=0.01, description='x2(0)',     style={'description_width': '95px'}),
    y1=     FloatSlider(value=0.1,  min=-2,  max=2,  step=0.01, description='target y1', style={'description_width': '95px'}),
    y2=     FloatSlider(value=-0.1, min=-2,  max=2,  step=0.01, description='target y2', style={'description_width': '95px'}),
    n=      IntSlider  (value=20,   min=5,   max=100,step=1,    description='steps',      style={'description_width': '95px'}),
    gamma=  FloatSlider(value=0.0,  min=0,   max=1,  step=0.01, description='friction γ', style={'description_width': '95px'}),
    epsilon=FloatSlider(value=0.0,  min=0,   max=1,  step=0.01, description='noise ε',    style={'description_width': '95px'}),
    tau=    FloatSlider(value=1.0,  min=0.01,max=2,  step=0.01, description='τ',          style={'description_width': '95px'})
);

---

## Bonus — Drift Field Visualization

Plot the drift field $v_1$ and $v_2$ as a function of $(x_1, x_2)$ for fixed targets. Arrows show the direction and magnitude of the drift at each point in the 2D state space.

In [9]:
def plot_vector_field(y1=0.5, y2=-0.5, tau=1.0, gamma=0.0):
    grid = np.linspace(-2, 2, 20)
    X1, X2 = np.meshgrid(grid, grid)

    V1 = np.vectorize(lambda a, b: v1(a, b, y1, y2, tau))(X1, X2)
    V2 = np.vectorize(lambda a, b: v2(a, b, y1, y2, tau))(X1, X2)
    V1 *= (1 - gamma)
    V2 *= (1 - gamma)

    speed = np.sqrt(V1**2 + V2**2)

    fig, ax = plt.subplots(figsize=(6.5, 6))
    q = ax.quiver(X1, X2, V1, V2, speed, cmap='plasma', alpha=0.85)
    plt.colorbar(q, ax=ax, label='speed')
    ax.scatter([y1], [y2], marker='*', s=250, color='gold', zorder=5, label=f'(y1, y2) = ({y1:.2f}, {y2:.2f})')
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.set_title(f'drift field  τ={tau:.2f}  γ={gamma:.2f}')
    ax.legend()
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

interact(plot_vector_field,
    y1=   FloatSlider(value=0.5,  min=-2,  max=2,  step=0.05, description='target y1', style={'description_width': '90px'}),
    y2=   FloatSlider(value=-0.5, min=-2,  max=2,  step=0.05, description='target y2', style={'description_width': '90px'}),
    tau=  FloatSlider(value=1.0,  min=0.05,max=2,  step=0.05, description='τ',         style={'description_width': '90px'}),
    gamma=FloatSlider(value=0.0,  min=0,   max=0.99,step=0.01,description='friction γ',style={'description_width': '90px'})
);

interactive(children=(FloatSlider(value=0.5, description='target y1', max=2.0, min=-2.0, step=0.05, style=Slid…